## Reading data

In [87]:
import numpy as np
import polars as pl
import glob
from enum import Enum

class CountMode(Enum):
    TOTAL = 0
    TWOBYONE = 1
    THREEBYONE = 2

def unrank(ranks, base=6, dims=12):
    ranks = np.array(ranks, dtype=np.int64)
    powers = base ** np.arange(dims-1,-1,-1)
    unranked = (ranks[:, None] // powers) % base
    return unranked

    
lazy_df = pl.scan_csv("part*.txt")
def cluster_filter_by_size(min, max):
    query = (
        lazy_df
        .filter(pl.col("Cluster Size").is_between(min, max))
        .group_by(pl.all())
        .agg(pl.len())
    )
    df_result = query.collect(streaming=True)
    return df_result

def cluster_filter_by_length(min, max):
    query = (
        lazy_df
        .filter(pl.col("Max Length").is_between(min, max))
        .group_by(pl.all())
        .agg(pl.len())
    )
    df_result = query.collect(streaming=True)
    return df_result

def cluster_filter_by_pieces(mode : CountMode, target_count : int):
    base = 6
    dims = 12
    dim_exprs = []

    for i in range(dims):
        power = base ** (dims - 1 - i)
        dim_expr = (pl.col("Cluster") // power) % base
        dim_exprs.append(dim_expr)
    
    count_1 = pl.sum_horizontal([expr == 1 for expr in dim_exprs])
    count_2 = pl.sum_horizontal([expr == 2 for expr in dim_exprs])
    count_3 = pl.sum_horizontal([expr == 3 for expr in dim_exprs])
    count_4 = pl.sum_horizontal([expr == 4 for expr in dim_exprs])
    count_5 = pl.sum_horizontal([expr == 5 for expr in dim_exprs])

    p_3x1 = count_1 + count_2 + count_3
    p_2x1 = count_2 + count_3 + count_4 + (2*count_5)

    if mode == CountMode.THREEBYONE:
        piece_count_expr = p_3x1
    elif mode == CountMode.TWOBYONE:
        piece_count_expr = p_2x1
    else:
        piece_count_expr = p_2x1 + p_3x1

    query = (
        lazy_df
        .with_columns(Cluster=pl.col("Cluster").str.replace("0x", ""))
        .with_columns(Cluster=pl.col("Cluster").str.to_integer(base=16))
        .filter(piece_count_expr == target_count)
        .group_by(pl.all())
        .agg(pl.len())
    )
    df_result = query.collect(streaming=True)
    df_result = df_result.with_columns(
        Cluster=pl.col("Cluster").map_elements(lambda x: hex(x), return_dtype=pl.String)
    )
    return df_result

def cluster_filter_by_exact_pieces(target_2x1: int, target_3x1: int):
    base = 6
    dims = 12
    dim_exprs = []

    for i in range(dims):
        power = base ** (dims - 1 - i)
        dim_expr = (pl.col("Cluster") // power) % base
        dim_exprs.append(dim_expr)

    count_1 = pl.sum_horizontal([expr == 1 for expr in dim_exprs])
    count_2 = pl.sum_horizontal([expr == 2 for expr in dim_exprs])
    count_3 = pl.sum_horizontal([expr == 3 for expr in dim_exprs])
    count_4 = pl.sum_horizontal([expr == 4 for expr in dim_exprs])
    count_5 = pl.sum_horizontal([expr == 5 for expr in dim_exprs])

    p_3x1 = count_1 + count_2 + count_3
    p_2x1 = count_2 + count_3 + count_4 + (2 * count_5)

    query = (
        lazy_df
        .with_columns(Cluster=pl.col("Cluster").str.replace("0x", "").str.to_integer(base=16))
        .filter((p_2x1 == target_2x1) & (p_3x1 == target_3x1))
        
        .group_by(pl.all())
        .agg(pl.len())
    )

    df_result = query.collect(streaming=True)
    df_result = df_result.with_columns(
        Cluster=pl.col("Cluster").map_elements(lambda x: hex(x), return_dtype=pl.String)
    )
    return df_result
    
def transform_clusters_for_input(df : pl.Series):
    base = 6
    if df.dtype == pl.String:
        df = df.str.replace("0x", "").str.to_integer(base=16)
    ranks = df.to_numpy()
    powers = base ** np.arange(11, -1, -1, dtype=np.uint64)
    matrix = (ranks[:, None] // powers) % base
    matrix = np.delete(matrix, 8, axis=1)
    new_powers = base ** np.arange(10, -1, -1, dtype=np.uint64)
    new_ranks = np.dot(matrix, new_powers)
    new_ranks = new_ranks.astype(int).tolist()
    new_ranks = [ hex(int(i)) for i in new_ranks]
    return new_ranks

In [12]:
df = cluster_filter_by_length(30, 1000)
print(df)

/tmp/ipykernel_548637/1294640368.py:23: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  df_result = query.collect(streaming=True)


shape: (1_797, 7)
┌────────────┬───────┬──────────────┬──────────────┬────────────┬───────┬─────┐
│ Cluster    ┆ Legal ┆ Has Solution ┆ Cluster Size ┆ Max Length ┆ Time  ┆ len │
│ ---        ┆ ---   ┆ ---          ┆ ---          ┆ ---        ┆ ---   ┆ --- │
│ str        ┆ i64   ┆ i64          ┆ i64          ┆ i64        ┆ f64   ┆ u32 │
╞════════════╪═══════╪══════════════╪══════════════╪════════════╪═══════╪═════╡
│ 0x596ef3c8 ┆ 1     ┆ 1            ┆ 5549         ┆ 30         ┆ 0.149 ┆ 1   │
│ 0x68712274 ┆ 1     ┆ 1            ┆ 7886         ┆ 30         ┆ 0.465 ┆ 1   │
│ 0x190ed588 ┆ 1     ┆ 1            ┆ 3043         ┆ 30         ┆ 0.119 ┆ 1   │
│ 0x596bcf68 ┆ 1     ┆ 1            ┆ 10615        ┆ 31         ┆ 0.527 ┆ 1   │
│ 0x596bcf6c ┆ 1     ┆ 1            ┆ 23538        ┆ 35         ┆ 1.965 ┆ 1   │
│ …          ┆ …     ┆ …            ┆ …            ┆ …          ┆ …     ┆ …   │
│ 0x190b796c ┆ 1     ┆ 1            ┆ 5944         ┆ 43         ┆ 0.475 ┆ 1   │
│ 0x57837e89 ┆ 1     ┆

In [13]:
df = cluster_filter_by_size(200000, 600000)
print(df)

/tmp/ipykernel_548637/1294640368.py:13: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  df_result = query.collect(streaming=True)


shape: (298, 7)
┌────────────┬───────┬──────────────┬──────────────┬────────────┬───────┬─────┐
│ Cluster    ┆ Legal ┆ Has Solution ┆ Cluster Size ┆ Max Length ┆ Time  ┆ len │
│ ---        ┆ ---   ┆ ---          ┆ ---          ┆ ---        ┆ ---   ┆ --- │
│ str        ┆ i64   ┆ i64          ┆ i64          ┆ i64        ┆ f64   ┆ u32 │
╞════════════╪═══════╪══════════════╪══════════════╪════════════╪═══════╪═════╡
│ 0x6562d20c ┆ 1     ┆ 1            ┆ 262010       ┆ 10         ┆ 5.234 ┆ 1   │
│ 0x6752500c ┆ 1     ┆ 1            ┆ 246890       ┆ 6          ┆ 2.502 ┆ 1   │
│ 0x656499ac ┆ 1     ┆ 1            ┆ 270068       ┆ 8          ┆ 3.804 ┆ 1   │
│ 0x6564e08c ┆ 1     ┆ 1            ┆ 271925       ┆ 6          ┆ 2.949 ┆ 1   │
│ 0x675417a9 ┆ 1     ┆ 1            ┆ 200770       ┆ 5          ┆ 0.243 ┆ 1   │
│ …          ┆ …     ┆ …            ┆ …            ┆ …          ┆ …     ┆ …   │
│ 0x26ea6164 ┆ 1     ┆ 1            ┆ 200770       ┆ 6          ┆ 1.849 ┆ 1   │
│ 0x67cc0cb0 ┆ 1     ┆ 1

In [55]:
df = cluster_filter_by_pieces(CountMode.TOTAL, 2)
print(df)

/tmp/ipykernel_548637/1464831295.py:73: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  df_result = query.collect(streaming=True)


shape: (22, 7)
┌────────────┬───────┬──────────────┬──────────────┬────────────┬──────┬─────┐
│ Cluster    ┆ Legal ┆ Has Solution ┆ Cluster Size ┆ Max Length ┆ Time ┆ len │
│ ---        ┆ ---   ┆ ---          ┆ ---          ┆ ---        ┆ ---  ┆ --- │
│ str        ┆ i64   ┆ i64          ┆ i64          ┆ i64        ┆ f64  ┆ u32 │
╞════════════╪═══════╪══════════════╪══════════════╪════════════╪══════╪═════╡
│ 0x361      ┆ 1     ┆ 1            ┆ 20           ┆ 1          ┆ 0.0  ┆ 1   │
│ 0x159fdb60 ┆ 1     ┆ 1            ┆ 17           ┆ 1          ┆ 0.0  ┆ 1   │
│ 0x364      ┆ 1     ┆ 1            ┆ 25           ┆ 1          ┆ 0.0  ┆ 1   │
│ 0x366      ┆ 1     ┆ 1            ┆ 20           ┆ 1          ┆ 0.0  ┆ 1   │
│ 0x99c960   ┆ 1     ┆ 1            ┆ 14           ┆ 2          ┆ 0.0  ┆ 1   │
│ …          ┆ …     ┆ …            ┆ …            ┆ …          ┆ …    ┆ …   │
│ 0xe6a9360  ┆ 1     ┆ 1            ┆ 21           ┆ 1          ┆ 0.0  ┆ 1   │
│ 0x567f6360 ┆ 1     ┆ 1            ┆

In [58]:
df = cluster_filter_by_exact_pieces(1, 1)
print(df)

/tmp/ipykernel_548637/1267056461.py:107: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  df_result = query.collect(streaming=True)


shape: (11, 7)
┌───────────┬───────┬──────────────┬──────────────┬────────────┬──────┬─────┐
│ Cluster   ┆ Legal ┆ Has Solution ┆ Cluster Size ┆ Max Length ┆ Time ┆ len │
│ ---       ┆ ---   ┆ ---          ┆ ---          ┆ ---        ┆ ---  ┆ --- │
│ str       ┆ i64   ┆ i64          ┆ i64          ┆ i64        ┆ f64  ┆ u32 │
╞═══════════╪═══════╪══════════════╪══════════════╪════════════╪══════╪═════╡
│ 0x366     ┆ 1     ┆ 1            ┆ 20           ┆ 1          ┆ 0.0  ┆ 1   │
│ 0x21c0    ┆ 1     ┆ 1            ┆ 20           ┆ 1          ┆ 0.0  ┆ 1   │
│ 0x99c960  ┆ 1     ┆ 1            ┆ 14           ┆ 2          ┆ 0.0  ┆ 1   │
│ 0x19a460  ┆ 1     ┆ 1            ┆ 14           ┆ 2          ┆ 0.0  ┆ 1   │
│ 0xb9a0    ┆ 1     ┆ 1            ┆ 17           ┆ 2          ┆ 0.0  ┆ 1   │
│ …         ┆ …     ┆ …            ┆ …            ┆ …          ┆ …    ┆ …   │
│ 0x870     ┆ 1     ┆ 1            ┆ 20           ┆ 1          ┆ 0.0  ┆ 1   │
│ 0x448e0   ┆ 1     ┆ 1            ┆ 14          

In [ ]:
import subprocess
list = transform_clusters_for_input(df["Cluster"])

with open("temp.txt", "w") as file:
    for cluster in list:
        print(cluster, file=file)

binary_path = "./retrograde_analysis/main -h > output_boards.txt"

result = subprocess.run(binary_path, shell=True)

FileNotFoundError: [Errno 2] No such file or directory: './retrograde_analysis/main -h > output_boards.txt'